In [1]:
from transformers import (
    AutoTokenizer, BertGenerationEncoder, BertGenerationDecoder,
    DataCollatorWithPadding, DataCollatorForSeq2Seq,
    TrainingArguments, Trainer, AutoModelForSeq2SeqLM,
    EncoderDecoderModel
)
import torch
import pandas as pd
from datasets import Dataset
from torch.utils.data import DataLoader

In [2]:
chkpt_path = '/dccstor/chrisconst2/AutoQA/fmea_recommender/entity_masking_mlm/checkpoint-14000'

In [3]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
encoder = BertGenerationEncoder.from_pretrained(chkpt_path, bos_token_id=101, eos_token_id=102)
decoder = BertGenerationDecoder.from_pretrained(
    chkpt_path, add_cross_attention=True, is_decoder=True, bos_token_id=101, eos_token_id=102
)
bert2bert = EncoderDecoderModel(encoder=encoder, decoder=decoder)

You are using a model of type bert to instantiate a model of type bert-generation. This is not supported for all configurations of models and can yield errors.
You are using a model of type bert to instantiate a model of type bert-generation. This is not supported for all configurations of models and can yield errors.
Some weights of BertGenerationDecoder were not initialized from the model checkpoint at /dccstor/chrisconst2/AutoQA/fmea_recommender/entity_masking_mlm/checkpoint-14000 and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.c

In [4]:
sel_cols = ['assetlongdescription_entity_llms', 'failurelocation_original', 
            'assetlongdescription_original', 'mode']
df = pd.read_csv('processed/asset2item.csv')[sel_cols]
df.rename({'assetlongdescription_original': 'answers.text'}, axis=1, inplace=True)
df['answers.text'] = pd.Series(df['answers.text'], dtype="string")
df.failurelocation_original = df.failurelocation_original.apply(lambda x: str(list(eval(x))))

In [5]:
df['question'] = 'What are the failure locations?'

In [6]:
df.rename({
    'answers.text': 'context',
    'failurelocation_original': 'answer'
}, axis=1, inplace=True)

In [7]:
df_train = df[df['mode']=='train']
df_val = df[df['mode']=='val']
df_test = df[df['mode']=='test']

In [8]:
ds_train = Dataset.from_pandas(df_train)
ds_val = Dataset.from_pandas(df_val)
ds_test = Dataset.from_pandas(df_test)

In [9]:
def collate_fn(data):
    input_text = [item['context'] for item in data]
    label_text = [item['answer'] for item in data]
    question_text = [item['question'] for item in data]
    tokenized_input = tokenizer(input_text, question_text, return_tensors='pt', truncation=True,
                                max_length=512, padding='max_length')
    label_ids = tokenizer(label_text, return_tensors='pt', truncation=True,
                          max_length=512, padding='max_length')['input_ids']
    return {
        'input_ids': tokenized_input['input_ids'],
        'decoder_input_ids': label_ids,
        'labels': label_ids,
        'attention_mask': tokenized_input['attention_mask']
    }

In [10]:
dl = DataLoader(ds_val, batch_size=8, collate_fn=collate_fn)

In [11]:
item = next(iter(dl))

In [1]:
# bert2bert(**item)

In [22]:
print("Question + context" + "*"*90)
question = tokenizer.decode(item['input_ids'][0]).replace('[PAD]', '')
print(question)
print("Answer" + "*"*100)
answer = tokenizer.decode(item['labels'][0]).replace('[PAD]', '')
print(answer)

Question + context******************************************************************************************
[CLS] the equipment battery - charger, is categorized as electrical asset and has the following boundary : the boundary of a typical battery charger for the purpose of this database is defined to include the following :, battery charger input breakers are excluded, because pm for these can be found by referring to motor control centers. note, this program assumes that the battery charger is in nominally good condition to begin with. battery chargers that have not been serviced for a long time may need to have a detailed inspection performed before this program is applied. [SEP] what are the failure locations? [SEP]                                                                                                                                                                                                                                                                             

In [12]:
training_args = TrainingArguments(
    output_dir="finetune_qa",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    remove_unused_columns=False
    
)

trainer = Trainer(
    model=bert2bert,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=collate_fn
)

trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_config.py:322: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/transformers/mo

Epoch,Training Loss,Validation Loss



KeyboardInterrupt

